# PHYT1D — Module 08: Evaluation Metrics

Implements all evaluation metrics from Section 12 of the PHYT1D proposal:

- **MARD** — Mean Absolute Relative Difference [%] (pass < 10%)
- **MAPE** — Mean Absolute Percentage Error [%]
- **gRMSE** — Glucose Root Mean Square Error [mg/dL] (replaces RMSE)
- **TIR / TBR / TAR** — Time in Range metrics (Battelino 2019)
- **Clarke Error Grid Analysis** (Clarke 1987)
- **Parameter recovery RE**
- **Posterior coverage**
- **Wilcoxon signed-rank test** (Bonferroni corrected)


In [ ]:
import numpy as np
from scipy import stats
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import matplotlib.patches as patches


## 8.0 Grid Alignment Utility

In [ ]:
def align_traces(ig_true, ig_sim, times_true=None, times_sim=None):
    """
    Align ig_sim onto ig_true's time grid so metric functions always
    receive arrays of identical length.

    Two modes
    ---------
    1. With time arrays (recommended):
       Linearly interpolates ig_sim at the timestamps in times_true.
       Works correctly even when the solver used adaptive step sizes.

    2. Without time arrays:
       Resamples ig_sim uniformly to len(ig_true) via a unit-interval
       linear interpolant.  Assumes both traces span the same duration.

    Parameters
    ----------
    ig_true    : array-like (n,)   ground-truth IG [mg/dL]
    ig_sim     : array-like (m,)   simulated  IG [mg/dL]
    times_true : array-like (n,) | None  timestamps for ig_true [any unit]
    times_sim  : array-like (m,) | None  timestamps for ig_sim  [same unit]

    Returns
    -------
    ig_true_a : np.ndarray (n,)
    ig_sim_a  : np.ndarray (n,)  — ig_sim interpolated onto ig_true grid
    """
    ig_true = np.asarray(ig_true, dtype=float)
    ig_sim  = np.asarray(ig_sim,  dtype=float)

    if len(ig_true) == len(ig_sim):
        return ig_true, ig_sim

    if times_true is not None and times_sim is not None:
        times_true = np.asarray(times_true, dtype=float)
        times_sim  = np.asarray(times_sim,  dtype=float)
        f = interp1d(times_sim, ig_sim, kind="linear",
                     bounds_error=False, fill_value="extrapolate")
        return ig_true, f(times_true)
    else:
        # Uniform resample — assumes identical total duration
        x_sim  = np.linspace(0, 1, len(ig_sim))
        x_true = np.linspace(0, 1, len(ig_true))
        f = interp1d(x_sim, ig_sim, kind="linear",
                     bounds_error=False, fill_value="extrapolate")
        return ig_true, f(x_true)


## 8.1 MARD — Mean Absolute Relative Difference

In [ ]:
def MARD(ig_true, ig_sim):
    """
    MARD = (1 / (100·D)) · Σ_p Σ_k |IG_obs(t) − IG_hat(t)| / IG_obs(t)

    For a single patient:
    MARD = mean(|IG_true − IG_sim| / IG_true) × 100   [%]

    Pass threshold: MARD < 10% (approximates CGM device accuracy).

    Parameters
    ----------
    ig_true : array [mg/dL] — ground-truth IG
    ig_sim  : array [mg/dL] — simulated IG (same time grid)

    Returns
    -------
    float — MARD [%]
    """
    ig_true = np.asarray(ig_true, dtype=float)
    ig_sim  = np.asarray(ig_sim,  dtype=float)
    return float(np.mean(np.abs(ig_true - ig_sim) / ig_true) * 100.0)


def MARD_population(ig_true_all, ig_sim_all):
    """
    Population-level MARD averaged across n_subjects.

    Parameters
    ----------
    ig_true_all : array (n_subjects, n_timepoints)
    ig_sim_all  : array (n_subjects, n_timepoints)

    Returns
    -------
    float — mean MARD [%]
    """
    mards = [MARD(ig_true_all[p], ig_sim_all[p]) for p in range(len(ig_true_all))]
    return float(np.mean(mards)), mards


## 8.2 MAPE & gRMSE

In [ ]:
def MAPE(ig_true, ig_sim):
    """
    MAPE = mean(|IG_true − IG_sim| / IG_true) × 100   [%]

    Similar to MARD but without the normalisation convention used in the
    PHYT1D proposal.  MAPE is the standard ML/forecasting definition and
    makes model-to-literature comparison straightforward.

    Parameters
    ----------
    ig_true : array [mg/dL] — ground-truth IG
    ig_sim  : array [mg/dL] — simulated IG (same time grid)

    Returns
    -------
    float — MAPE [%]
    """
    ig_true = np.asarray(ig_true, dtype=float)
    ig_sim  = np.asarray(ig_sim,  dtype=float)
    return float(np.mean(np.abs(ig_true - ig_sim) / ig_true) * 100.0)


def MAPE_population(ig_true_all, ig_sim_all):
    """Population-level MAPE averaged across n_subjects.

    Parameters
    ----------
    ig_true_all : array (n_subjects, n_timepoints)
    ig_sim_all  : array (n_subjects, n_timepoints)

    Returns
    -------
    float — mean MAPE [%]
    list  — per-subject MAPE values
    """
    mapes = [MAPE(ig_true_all[p], ig_sim_all[p]) for p in range(len(ig_true_all))]
    return float(np.mean(mapes)), mapes


# ─────────────────────────────────────────────────────────────────────────────

def gRMSE(ig_true, ig_sim):
    """
    Glucose RMSE (gRMSE) — penalises hypoglycaemic errors more heavily
    than hyperglycaemic ones via the Clarke penalty function P(g):

        P(g) = 10 · log(g / 18)^2    (Clarke 1987 linearisation)

        gRMSE = √[ mean( (P(IG_sim) − P(IG_true))² ) ]

    This transforms raw glucose values onto a risk scale before computing
    RMSE, giving clinically appropriate weighting to dangerous low events.

    Parameters
    ----------
    ig_true : array [mg/dL]
    ig_sim  : array [mg/dL]

    Returns
    -------
    float — gRMSE (risk-weighted, dimensionless)
    """
    ig_true = np.asarray(ig_true, dtype=float)
    ig_sim  = np.asarray(ig_sim,  dtype=float)

    def _risk(g):
        """Clarke log-risk transformation."""
        return 10.0 * np.log(np.clip(g, 1e-6, None) / 18.0) ** 2

    return float(np.sqrt(np.mean((_risk(ig_sim) - _risk(ig_true)) ** 2)))


def gRMSE_population(ig_true_all, ig_sim_all):
    """Population-level gRMSE.

    Parameters
    ----------
    ig_true_all : array (n_subjects, n_timepoints)
    ig_sim_all  : array (n_subjects, n_timepoints)

    Returns
    -------
    float — mean gRMSE
    list  — per-subject gRMSE values
    """
    grmses = [gRMSE(ig_true_all[p], ig_sim_all[p]) for p in range(len(ig_true_all))]
    return float(np.mean(grmses)), grmses


## 8.3 Time-in-Range Metrics (Battelino 2019)

In [ ]:
def time_in_range(ig, dt=1.0):
    """
    Compute TIR, TBR, TAR from an IG trace.

    TIR = % time 70 ≤ IG ≤ 180 mg/dL  (target: > 70%)
    TBR = % time IG < 70 mg/dL          (target: < 4%)
    TAR = % time IG > 180 mg/dL         (target: < 25%)

    Parameters
    ----------
    ig : array [mg/dL]
    dt : float — time step [min]

    Returns
    -------
    dict — {TIR, TBR, TAR} in %
    """
    ig   = np.asarray(ig, dtype=float)
    n    = len(ig)
    tir  = np.sum((ig >= 70) & (ig <= 180)) / n * 100.0
    tbr  = np.sum(ig < 70)                  / n * 100.0
    tar  = np.sum(ig > 180)                 / n * 100.0
    return {"TIR": round(tir, 1), "TBR": round(tbr, 1), "TAR": round(tar, 1)}


def time_in_range_population(ig_all, dt=1.0):
    """Population-level TIR/TBR/TAR summary."""
    results = [time_in_range(ig, dt) for ig in ig_all]
    summary = {}
    for key in ["TIR","TBR","TAR"]:
        vals = [r[key] for r in results]
        summary[key] = {"mean": np.mean(vals), "std": np.std(vals), "all": vals}
    return summary


## 8.4 Clarke Error Grid Analysis

In [ ]:
def clarke_error_grid(ig_ref, ig_sim, ax=None):
    """
    Clarke Error Grid Analysis (Clarke 1987).
    Classifies (reference, estimate) pairs into zones A–E.

    Zone A: clinically accurate (≤ 20% deviation or both < 70 mg/dL)
    Zone B: benign errors (>20% but safe outcome)
    Zone C–E: increasingly dangerous errors

    Parameters
    ----------
    ig_ref : array — reference glucose [mg/dL]
    ig_sim : array — estimated glucose [mg/dL]
    ax     : matplotlib Axes | None

    Returns
    -------
    zone_counts : dict — {A:%, B:%, C:%, D:%, E:%}
    """
    ig_ref = np.asarray(ig_ref, dtype=float)
    ig_sim = np.asarray(ig_sim, dtype=float)
    n = len(ig_ref)

    zones = {"A":0, "B":0, "C":0, "D":0, "E":0}
    labels = []

    for ref, sim in zip(ig_ref, ig_sim):
        # Zone A: within 20% of reference or both ≤ 70
        if (ref <= 70 and sim <= 70) or abs(sim - ref)/ref <= 0.20:
            zones["A"] += 1; labels.append("A")
        # Zone E (dangerous errors — opposite clinical action)
        elif (ref <= 70 and sim >= 180) or (ref >= 180 and sim <= 70):
            zones["E"] += 1; labels.append("E")
        # Zone C
        elif (ref >= 70 and sim >= ref*1.20 + 20) or (ref <= 70 and sim >= 160):
            zones["C"] += 1; labels.append("C")
        # Zone D
        elif (ref >= 240 and sim <= 70) or (ref <= 70 and 70 < sim < 180):
            zones["D"] += 1; labels.append("D")
        else:
            zones["B"] += 1; labels.append("B")

    zone_pct = {k: v/n*100 for k, v in zones.items()}

    if ax is not None:
        colors = {"A":"#2ecc71","B":"#f1c40f","C":"#e67e22","D":"#e74c3c","E":"#8e44ad"}
        scatter_colors = [colors[l] for l in labels]
        ax.scatter(ig_ref, ig_sim, c=scatter_colors, alpha=0.4, s=10, zorder=3)
        ax.plot([0,400],[0,400],"k--",lw=1,alpha=0.5)
        ax.set_xlim(0,400); ax.set_ylim(0,400)
        ax.set_xlabel("Reference IG [mg/dL]",fontsize=11)
        ax.set_ylabel("Estimated IG [mg/dL]",fontsize=11)
        ax.set_title("Clarke Error Grid",fontsize=12)
        legend_patches = [patches.Patch(color=c, label=f"Zone {z}: {zone_pct[z]:.1f}%")
                          for z, c in colors.items()]
        ax.legend(handles=legend_patches, fontsize=9, loc="upper left")

    return zone_pct


def print_zone_summary(zone_pct):
    print("\n── Clarke Error Grid Zones ─────────────────────────────────")
    for z, pct in zone_pct.items():
        bar = "█" * int(pct / 2)
        print(f"  Zone {z}: {pct:5.1f}%  {bar}")


## 8.5 Parameter Recovery

In [ ]:
def parameter_recovery_RE(theta_hat, theta_true, threshold=0.20):
    """
    Relative Error: RE = |θ̂ − θ*| / θ*  per parameter.
    Pass threshold: RE < 20% per Group A parameter.

    Parameters
    ----------
    theta_hat  : dict — posterior median estimates
    theta_true : dict — ground-truth parameters
    threshold  : float

    Returns
    -------
    dict — {param: (RE, passed)}
    """
    print("\n── Parameter Recovery (RE < 20%) ───────────────────────────")
    report = {}
    for name in theta_hat:
        if name not in theta_true: continue
        re = abs(theta_hat[name] - theta_true[name]) / abs(theta_true[name])
        ok = re < threshold
        report[name] = (re, ok)
        print(f"  {name:8s}  RE={re*100:6.1f}%  {'✓' if ok else '✗'}")
    return report


def posterior_coverage(chain, theta_true, low=25, high=75):
    """
    Posterior coverage: fraction of subjects where θ* ∈ [25th, 75th] CI.
    Target: ≥ 50%.

    For a single subject: True if θ* in posterior CI.
    """
    param_names = list(chain[0].keys())
    results = {}
    for name in param_names:
        if name not in theta_true: continue
        samples = [s[name] for s in chain]
        lo = np.percentile(samples, low)
        hi = np.percentile(samples, high)
        covered = lo <= theta_true[name] <= hi
        results[name] = covered
    return results


## 8.6 Wilcoxon Signed-Rank Test

In [ ]:
def wilcoxon_test(ig_true_all, ig_sim_all, dt=1.0, alpha=0.05):
    """
    Wilcoxon signed-rank test on per-subject MARD, Bonferroni corrected.

    Parameters
    ----------
    ig_true_all : list of arrays
    ig_sim_all  : list of arrays
    dt          : float
    alpha       : float — family-wise error rate

    Returns
    -------
    dict — {statistic, p_value_raw, p_value_corrected, significant}
    """
    mards = [MARD(ig_true_all[p], ig_sim_all[p]) for p in range(len(ig_true_all))]
    threshold_arr = [10.0] * len(mards)    # null: MARD == 10%
    stat, p_raw = stats.wilcoxon(mards, threshold_arr, alternative="less")
    p_corr = min(p_raw * len(mards), 1.0)  # Bonferroni
    sig    = p_corr < alpha

    print(f"\n── Wilcoxon Signed-Rank Test (Bonferroni) ──────────────────")
    print(f"  stat={stat:.2f}  p_raw={p_raw:.4f}  p_corrected={p_corr:.4f}  "
          f"significant={'YES' if sig else 'NO'}")
    return {"statistic": stat, "p_raw": p_raw, "p_corrected": p_corr, "significant": sig}


## 8.7 Full Evaluation Report

In [ ]:
def full_evaluation_report(ig_true_all, ig_sim_all, theta_hat=None, theta_true=None,
                           chain=None, dt=1.0,
                           times_true=None, times_sim=None):
    """
    Run all evaluation metrics and print a summary report.

    Handles mismatched time grids automatically via align_traces().

    Parameters
    ----------
    ig_true_all : array (n_subjects, n_timesteps_true)  or list of arrays
    ig_sim_all  : array (n_subjects, n_timesteps_sim)   or list of arrays
    theta_hat   : dict | None
    theta_true  : dict | None
    chain       : list | None
    dt          : float — time step [min] used for TIR (after alignment)
    times_true  : array (n_timesteps_true,) | None — shared time grid for ig_true
    times_sim   : array (n_timesteps_sim,)  | None — shared time grid for ig_sim

    Returns
    -------
    report : dict
    """
    print("\n" + "="*60)
    print("  PHYT1D — Full Evaluation Report")
    print("="*60)

    # ── align every subject's traces onto the same grid ──────────────
    ig_true_aligned = []
    ig_sim_aligned  = []
    for p in range(len(ig_true_all)):
        t_a, s_a = align_traces(
            ig_true_all[p], ig_sim_all[p],
            times_true=times_true,
            times_sim=times_sim,
        )
        ig_true_aligned.append(t_a)
        ig_sim_aligned.append(s_a)

    mard_mean,  mards  = MARD_population(ig_true_aligned, ig_sim_aligned)
    mape_mean,  mapes  = MAPE_population(ig_true_aligned, ig_sim_aligned)
    grmse_mean, grmses = gRMSE_population(ig_true_aligned, ig_sim_aligned)
    tir_summary        = time_in_range_population(ig_sim_aligned, dt)

    print(f"\n  MARD  : {mard_mean:.2f}%  "
          f"({'PASS ✓' if mard_mean < 10 else 'FAIL ✗'})")
    print(f"  MAPE  : {mape_mean:.2f}%")
    print(f"  gRMSE : {grmse_mean:.4f}  (risk-weighted)")
    print(f"  TIR   : {tir_summary['TIR']['mean']:.1f}% ± {tir_summary['TIR']['std']:.1f}%")
    print(f"  TBR   : {tir_summary['TBR']['mean']:.1f}% ± {tir_summary['TBR']['std']:.1f}%")
    print(f"  TAR   : {tir_summary['TAR']['mean']:.1f}% ± {tir_summary['TAR']['std']:.1f}%")

    report = {
        "MARD_mean":  mard_mean,  "MARD_all":  mards,
        "MAPE_mean":  mape_mean,  "MAPE_all":  mapes,
        "gRMSE_mean": grmse_mean, "gRMSE_all": grmses,
        "TIR":        tir_summary,
        "passed":     mard_mean < 10.0,
    }

    if theta_hat and theta_true:
        re_report = parameter_recovery_RE(theta_hat, theta_true)
        report["param_RE"] = re_report

    if chain and theta_true:
        cov = posterior_coverage(chain, theta_true)
        n_covered = sum(cov.values())
        print(f"\n  Posterior coverage: {n_covered}/{len(cov)} params in 25–75th CI")
        report["posterior_coverage"] = cov

    return report


# ── Single-subject convenience wrapper ────────────────────────────────────────

def print_eval_report(ig_true, ig_sim, theta_hat=None, theta_true=None,
                      chain_A=None, chain_B=None,
                      times_true=None, times_sim=None):
    """
    Single-subject wrapper around full_evaluation_report.
    Accepts 1-D arrays and optional time vectors for grid alignment.

    Parameters
    ----------
    ig_true    : array (n,)  — ground-truth IG
    ig_sim     : array (m,)  — simulated IG  (may differ in length)
    theta_hat  : dict | None
    theta_true : dict | None
    chain_A    : list | None — posterior chain (passed as `chain`)
    chain_B    : list | None — unused here, kept for API compatibility
    times_true : array (n,) | None
    times_sim  : array (m,) | None

    Returns
    -------
    report : dict
    """
    return full_evaluation_report(
        ig_true_all=[ig_true],
        ig_sim_all=[ig_sim],
        theta_hat=theta_hat,
        theta_true=theta_true,
        chain=chain_A,
        times_true=times_true,
        times_sim=times_sim,
    )
